## Marine heatwave visualization

Interactive Plotly map of the NW Mediterranean bounding box, with a time slider. Sea surface temperature is shown on a blue intensity scale for normal conditions; cells flagged as part of a detected marine heatwave event switch to red, overriding the temperature-based color.

Starting with a small prototype (a couple of months) to validate the visual style, before scaling to the full 2016-2026 period at weekly or 5-day resolution.

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go

con = duckdb.connect()

In [2]:
# Look at the raw temperature data structure again, refresh
raw_sample = con.execute("""
    SELECT * FROM '../data/processed/med_sst_2016_2026.parquet' 
    LIMIT 5
""").df()
print(raw_sample)

        time   latitude  longitude  analysed_sst  __index_level_0__
0 2016-01-01  40.507652   2.043520    288.579994                  0
1 2016-01-01  40.507652   2.093567    288.649994                  1
2 2016-01-01  40.507652   2.143612    288.699994                  2
3 2016-01-01  40.507652   2.193659    288.739994                  3
4 2016-01-01  40.507652   2.243704    288.749994                  4


In [3]:
# Look at the events table structure
events_sample = con.execute("""
    SELECT * FROM '../data/processed/mhw_events.parquet' 
    LIMIT 5
""").df()
print(events_sample)

    latitude  longitude  group_id event_start  event_end  duration_days
0  41.009228  11.902573         0  2016-01-01 2016-01-06              6
1  40.758438   6.547657         0  2016-01-01 2016-01-05              5
2  40.758438  12.753355         0  2016-01-01 2016-01-05              5
3  40.507652   6.897978         0  2016-01-01 2016-01-05              5
4  40.858757  12.503124         0  2016-01-01 2016-01-05              5


In [4]:
# Test the interval JOIN for a small period (Jan-Mar 2016), which contains the verified 46-day event.
# Flags each cell/day as 'Y' if it falls inside a detected heatwave event, 'N' otherwise.
check_months = con.execute("""
    SELECT 
        raw.time,
        raw.latitude,
        raw.longitude,
        raw.analysed_sst,
        CASE WHEN events.event_start IS NOT NULL THEN 'Y' ELSE 'N' END AS in_heatwave
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    LEFT JOIN '../data/processed/mhw_events.parquet' AS events
        ON raw.latitude = events.latitude 
        AND raw.longitude = events.longitude
        AND raw.time BETWEEN events.event_start AND events.event_end
    WHERE raw.time BETWEEN '2016-01-01' AND '2016-03-31'
""").df()
pd.set_option('display.float_format', '{:.2f}'.format)
print(check_months.describe())

                                time   latitude  longitude  analysed_sst
count                        1086176 1086176.00 1086176.00    1086176.00
mean   2016-02-15 00:00:00.000000256      42.06       7.81        287.10
min              2016-01-01 00:00:00      40.51       2.04        279.83
25%              2016-01-23 00:00:00      41.16       5.20        286.62
50%              2016-02-15 00:00:00      41.91       7.75        287.10
75%              2016-03-09 00:00:00      42.87      10.15        287.62
max              2016-03-31 00:00:00      44.47      13.95        290.73
std                              NaN       1.04       3.07          0.94


In [5]:
# Check the balance of flagged vs unflagged rows: 'Y' should be a small minority,
# concentrated in the cells and dates covered by the verified Jan-Feb 2016 event.
print(check_months["in_heatwave"].value_counts())

in_heatwave
N    1041939
Y      44237
Name: count, dtype: int64


In [6]:
# How many distinct cells have at least one 'Y' day in this period?
n_cells_flagged = check_months[check_months["in_heatwave"] == "Y"][["latitude", "longitude"]].drop_duplicates().shape[0]
print(f"Distinct cells with at least one heatwave day: {n_cells_flagged}")

Distinct cells with at least one heatwave day: 1858


**Results**: 44,237 flagged rows across the test period, spread over 1,858 distinct cells, roughly 15% of the bounding box's sea cells. Initially this seemed high compared to the previously verified event (75 cells sharing the same Jan 1 - Feb 15 2016 window, per notebook 03), but it makes physical sense: marine heatwaves are not point phenomena, they cover contiguous patches of ocean at synoptic scale, so a real event naturally touches hundreds or thousands of neighboring cells, not an isolated one.

## First static frame: single day test

Before building the full animated slider, testing the visual style (blue intensity for normal SST, red override for heatwave cells) on a single day known to be inside the verified event.

In [7]:
# Extract a single day's data for the visual test
day_test = check_months[check_months["time"] == "2016-01-20"].copy()
print(day_test.shape)
print(day_test["in_heatwave"].value_counts())

(11936, 5)
in_heatwave
N    11349
Y      587
Name: count, dtype: int64


In [8]:
# Assign color: red if in_heatwave, otherwise a blue shade based on temperature
fig = go.Figure()

normal_cells = day_test[day_test["in_heatwave"] == "N"]
heatwave_cells = day_test[day_test["in_heatwave"] == "Y"]

fig.add_trace(go.Scattergeo(
    lon=normal_cells["longitude"],
    lat=normal_cells["latitude"],
    mode="markers",
    marker=dict(
        size=4,
        color=normal_cells["analysed_sst"],
        colorscale="Blues",
        showscale=True,
        colorbar=dict(title="SST (K)")
    ),
    name="Normal"
))

fig.add_trace(go.Scattergeo(
    lon=heatwave_cells["longitude"],
    lat=heatwave_cells["latitude"],
    mode="markers",
    marker=dict(size=4, color="red"),
    name="Heatwave"
))

fig.update_geos(
    lonaxis_range=[2, 14],
    lataxis_range=[40.5, 44.5],
    showcoastlines=True,
    showland=True
)

fig.update_layout(title="SST and detected heatwave cells — 2016-01-20", height=600)
fig.show()